In [1]:

import glob
import pandas as pd
import os
import pyarrow as pa
import numpy as np
import pyarrow.parquet as pq
import re


from scipy.stats import spearmanr, pearsonr


In [2]:
def format_corr_with_stars(corr, p):
    if pd.isna(corr) or pd.isna(p):
        return ''
    if p < 0.001:
        stars = '***'
    elif p < 0.01:
        stars = '**'
    elif p < 0.05:
        stars = '*'
    else:
        stars = ''
    return f"{corr:.2f}{stars}"

def get_background_color_from_corr(val):
    match = re.match(r"([+-]?\d+\.\d+)", str(val))
    if not match:
        return 'background-color: white'
    try:
        corr = float(match.group(1))
    except ValueError:
        return 'background-color: white'
    abs_corr = max(0.0, min(1.0, abs(corr)))
    DARK_R, DARK_G, DARK_B = 50, 150, 200
    WHITE = 255
    r = int(WHITE - abs_corr * (WHITE - DARK_R))
    g = int(WHITE - abs_corr * (WHITE - DARK_G))
    b = int(WHITE - abs_corr * (WHITE - DARK_B))
    return f'background-color: rgb({r}, {g}, {b})'

def get_text_color_for_contrast(val):
    match = re.match(r"([+-]?\d+\.\d+)", str(val))
    if not match:
        return 'color: black'
    try:
        corr = float(match.group(1))
    except ValueError:
        return 'color: black'
    return 'color: white' if abs(corr) >= 0.6 else 'color: black'

def get_background_color_from_corr(val):
    match = re.match(r"([+-]?\d+\.\d+)", str(val))
    if not match:
        return 'background-color: white'
    try:
        corr = float(match.group(1))
    except ValueError:
        return 'background-color: white'
    abs_corr = max(0.0, min(1.0, abs(corr)))
    DARK_R, DARK_G, DARK_B = 50, 150, 200
    WHITE = 255
    r = int(WHITE - abs_corr * (WHITE - DARK_R))
    g = int(WHITE - abs_corr * (WHITE - DARK_G))
    b = int(WHITE - abs_corr * (WHITE - DARK_B))
    return f'background-color: rgb({r}, {g}, {b})'

def get_text_color_for_contrast(val):
    match = re.match(r"([+-]?\d+\.\d+)", str(val))
    if not match:
        return 'color: black'
    try:
        corr = float(match.group(1))
    except ValueError:
        return 'color: black'
    return 'color: white' if abs(corr) >= 0.6 else 'color: black'

In [3]:

def rename_model(x):
       if x == "OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42":
              return "Olmo2-1B"
       
def rename_estimator(x):
      return x.split(":")[0]
def rename_linear_coder(x):
      return x.replace("Coder","").replace("Thresh","")
def extract_seed(x):
       return int(re.search(r'seed (\d+)', x).group(1)) if "seed" in x else None

def rename_random(x):
    return re.sub(r' with seed \d+', "", x)



df_validation = pq.ParquetDataset("results/validation").read().to_pandas()
df_validation["model"]  = df_validation["model"].apply(rename_model)
df_validation["estimator"]  = df_validation["estimator"].apply(rename_estimator)


In [4]:
def extract_k(explanation_type):
    if "The test instance (as a sanity check)" in explanation_type:
        return 1
    first_number_match = re.match(r"^\s*(\d+)\b", explanation_type)
    if first_number_match:
        return int(first_number_match.group(1))


    top_match = re.search(r"Top-(\d+)", explanation_type)
    if top_match:
        return int(top_match.group(1))

    random_match = re.search(r"(\d+)\s+random examples", explanation_type)
    if random_match:
        return int(random_match.group(1))

    return None

def replace_k(explanation_type, k):
    if k is None:
        return explanation_type
    # Only replace the first occurrence of the number as a standalone word
    return re.sub(rf"\b{k}\b", "X", explanation_type, count=1)

def vectorized_replace_k(explanation_types, ks):
    result = explanation_types.copy()
    for k in np.unique(ks[ks.notnull()]):  # only unique, non-null ks
        # Use string pattern, not compiled regex
        pattern = rf"\b{k}\b"
        mask = ks == k
        result.loc[mask] = result.loc[mask].str.replace(pattern, "X", n=1, regex=True)
    return result

def facility_location_hotfix(x):
    if ("facility" in x) and x.startswith("Top-"):
        return x[len("Top-"):]
    else:
        return x
def get_sort_type(x):
    for sort_type in ["scores with largest absolute value", "most positive scores", "most negative scores", "scores closest to zero"]:
        if sort_type in x:
            return sort_type
    return "-"





df_validation["k"] = df_validation["explanation_type"].apply(extract_k)
df_validation["explanation_type_no_k"] = vectorized_replace_k(df_validation["explanation_type"], df_validation["k"])
df_validation["explanation_type"] = df_validation["explanation_type"].apply(facility_location_hotfix) # hotfix: inconsistent naming scheme for facility location selections



df_validation["sort_type"] = df_validation["explanation_type"].apply(get_sort_type)

df_validation_random = df_validation[df_validation["explanation_type"].str.contains("random examples with seed")]
df_validation_selection = df_validation[~df_validation["explanation_type"].str.contains("random examples with seed")]



In [5]:
df_scoring = pq.ParquetDataset("results/scoring").read().to_pandas()

df_scoring["NMSE"] =1.0 / (df_scoring["pred_gain"] + 1e-8)
df_scoring["k"] = df_scoring["explanation_type"].apply(extract_k)
df_scoring["explanation_type_no_k"] = df_scoring.apply(
    lambda row: row["explanation_type"].replace(str(row["k"]), "X"),
    axis=1
)
df_scoring["explanation_type"] = df_scoring["explanation_type"].apply(facility_location_hotfix) # hotfix: inconsistent naming scheme for facility location selections
df_scoring["model"] = df_scoring["model"].apply(rename_model)
df_scoring["estimator"] = df_scoring["estimator"].apply(rename_estimator)
df_scoring["fl"] = df_scoring["explanation_type"].apply(lambda x: "FL" if "facility" in x else "N")
import re

def get_lambda(x):
    match = re.search(r'lambda=([\d.]+)', x)
    return float(match.group(1)) if match else "-"

df_scoring["lambda"] = df_scoring["explanation_type"].apply(get_lambda)
import re

def remove_lambda(expr: str):
    # remove substrings like lambda=0.5 or lambda = 0.1
    return re.sub(r'lambda\s*=\s*[\d.]+', 'lambda=X', expr)

df_scoring["explanation_type_no_lambda"] = (
    df_scoring["explanation_type"].apply(remove_lambda)
)


In [6]:
# pair selection and random validation scores
r = pd.merge(
                df_validation_selection,
                df_validation_random,
                on=[
                    "model",
                    "estimator",
                    "train_dataset",
                    "train_split",
                    "test_dataset",
                    "test_split",
                    "k",
                    "document_idx"
                ],
                suffixes=("_selection", "_random"),
                how="inner"
            )
r["validation_score_log_p"] = r["delta_log_p_selection"] - r["delta_log_p_random"]
r["validation_score_jsd"] = r["jsd_selection"] - r["jsd_random"]
r["validation_score_kld"] = r["kld(before||after)_selection"] - r["kld(before||after)_random"]



In [7]:
# merge with our score
rr = pd.merge(
    r,
    df_scoring,
    left_on=[
        "model",
        "estimator",
        "train_dataset",
        "train_split",
        "test_dataset",
        "test_split",
        "explanation_type_selection", 
        "k",
        "document_idx"
    ],
    right_on=[
        "model",
        "estimator",
        "train_dataset",
        "train_split",
        "test_dataset",
        "test_split",
        "explanation_type", 
        "k",
        "document_idx"
    ],
    suffixes=("_validation", "_scoring"),
    how="left"
)


rr_summary = rr.groupby([
    "estimator",
    "explanation_type_selection",
    "model",
    "train_dataset",
    "train_split",
    "test_dataset",
    "test_split",
    "k",
    "linear_coder",
    "explanation_type_no_k_selection",
    "fl",
    "lambda"
]).agg(
    mean_NMSE=("NMSE", "mean"),
    mean_validation_score_log_p=("validation_score_log_p", "mean"),
    count_validation_score_log_p=("validation_score_log_p", "count"),
    mean_validation_score_jsd=("validation_score_jsd", "mean"),
    count_validation_score_jsd=("validation_score_jsd", "count"),
    mean_validation_score_kld=("validation_score_kld", "mean"),
    count_validation_score_kld=("validation_score_kld", "count")
)

                
                      
                

In [17]:
def compute_and_export_correlations(rr, rr_summary, subset_filter=None, group_cols=None, sort_cols=None,
                                    row_label_cols=None, filename="./tables/validation.tex", caption="Correlation table"):
    """
    Compute correlations of NMSE vs validation scores and export to LaTeX.
    
    subset_filter: function to filter rr and rr_summary (takes DataFrame, returns filtered DataFrame)
    group_cols: list of columns to group by
    sort_cols: list of columns to sort by in the final table
    row_label_cols: list of columns to use as row labels (index) in the table
    """

    rr_subset = rr.copy()
    rr_summary_subset = rr_summary.reset_index()
    if subset_filter:
        rr_subset = subset_filter(rr_subset)
        rr_summary_subset = subset_filter(rr_summary_subset)


    rows = []
    for keys, df in rr_subset.groupby(group_cols):
        if len(df) < 2:
            rows.append({**dict(zip(group_cols, keys)),
                         "spearman_NMSE_log_p": "",
                         "spearman_NMSE_jsd": "",
          })
            continue
            
        from scipy.stats import rankdata
        mask = df["NMSE"].notna() & df["validation_score_log_p"].notna()
        x_rank = rankdata(df.loc[mask, "NMSE"])
    
        y_rank = rankdata(df.loc[mask, "validation_score_log_p"])
        s_logp_rho = np.corrcoef(x_rank, y_rank)[0, 1]
    
        y_rank = rankdata(df.loc[mask, "validation_score_jsd"])
        s_jsd_rho  = np.corrcoef(x_rank, y_rank)[0, 1]


        rows.append({**dict(zip(group_cols, keys)),
                     "spearman_NMSE_log_p": s_logp_rho,
                    "spearman_NMSE_jsd": s_jsd_rho,
             })

    pooled_df = pd.DataFrame(rows)

    if row_label_cols:
        pooled_df.set_index(row_label_cols, inplace=True)
    else:
        pooled_df.set_index(group_cols, inplace=True)

    correlation_cols = ["spearman_NMSE_log_p", "spearman_NMSE_jsd", ]
    hidden_cols = [col for col in pooled_df.columns if col not in correlation_cols]

    colnames = {
        "spearman_NMSE_log_p": "$\\rho (NMSE, \\log_p)$",
        "spearman_NMSE_jsd": "$\\rho (NMSE, JSD)$",
     
    }

    styled = (
        pooled_df.sort_values(by=sort_cols or correlation_cols, ascending=True, na_position="last")
                 .style
                 .map(get_background_color_from_corr, subset=correlation_cols)
                 .map(get_text_color_for_contrast, subset=correlation_cols)
                 .format_index(colnames.get, axis=1)
                 .hide(axis="columns", subset=hidden_cols)
    )

 
    display(styled)

    os.makedirs(os.path.dirname(filename), exist_ok=True)
    latex_tabular = styled.to_latex(convert_css=True, hrules=True,
                                   column_format='l' + 'l' * len(styled.data.columns))
    latex_table = (
        "\\begin{table}[htbp]\n"
        "\\scriptsize\n"
        f"\\caption{{{caption}}}\\label{{tab:validation}}\n"
        "\\centering\n"
        f"{latex_tabular}\n"
        "\\end{table}\n"
    )
    with open(filename, "w") as f:
        f.write(latex_table)





## per linear coder

In [18]:
for linear_coder in df_scoring["linear_coder"].unique():
    print("linear_coder",linear_coder)
    compute_and_export_correlations(
        rr, rr_summary,
        group_cols=["estimator","model","train_dataset","train_split","test_dataset","test_split","explanation_type","lambda","k"],
        subset_filter=lambda df: df[(df["lambda"]=="-")& (df["linear_coder"]==linear_coder)],
        row_label_cols=[ "k","estimator","explanation_type"],
        sort_cols=["k", "estimator","explanation_type"],
        filename=f"./tables/validation/per_linear_coder/{linear_coder}.tex",
        caption=f"Correlation of our score with validation scores for {linear_coder}"
    )

linear_coder MSECoderProjUSimp


linear_coder KLTCoder


linear_coder MSECoder


linear_coder MSECoderNNLSL2


linear_coder MSECoderProjUSimpSparse


linear_coder MSECoderProjUSimpSparseSoftThresh


## for MSECoderProjUSimp

In [19]:
linear_coder = "MSECoderProjUSimp"
compute_and_export_correlations(
    rr, rr_summary,
    subset_filter=lambda df: df[(df["linear_coder"]==linear_coder)],
    group_cols=["model","train_dataset","train_split","test_dataset","test_split",],
    sort_cols=[],
    row_label_cols=[],
    filename=f"./tables/validation/full/{linear_coder}.tex",
    caption=f"Correlation of our score with validation scores for {linear_coder}"
)



,,,,,"$\rho (NMSE, \log_p)$","$\rho (NMSE, JSD)$"
model,train_dataset,train_split,test_dataset,test_split,,
Olmo2-1B,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,train,loris3/tulu-3-sft-olmo-2-mixture-0225-sample,test,-0.130227,-0.115352


### mean per type

In [34]:

linear_coder = "MSECoderProjUSimp"
compute_and_export_correlations(
    rr, rr_summary,
    subset_filter=lambda df: df[(df["linear_coder"]==linear_coder)],
    group_cols=["estimator","model","train_dataset","train_split","test_dataset","test_split", "explanation_type_no_k"],
    sort_cols=["estimator", "explanation_type_no_k"],
    row_label_cols=[ "explanation_type_no_k","estimator"],
    filename=f"./tables/validation/per_selection_type/{linear_coder}.tex",
    caption=f"Per selection-type correlation of our score with validation scores for {linear_coder}"
)



,,"$\rho (NMSE, \log_p)$","$\rho (NMSE, JSD)$"
explanation_type_no_k,estimator,,
The test instance (as a sanity check),BM25Estimator,0.447092,-0.798973
Top-X least influential (scores closest to zero),BM25Estimator,0.005707,0.098617
Top-X most influential (scores with largest absolute value),BM25Estimator,-0.318511,-0.301782
X by facility location from Top-100 least influential (scores closest to zero). lambda=0.0,BM25Estimator,-0.005561,0.086593
X by facility location from Top-100 least influential (scores closest to zero). lambda=0.2X,BM25Estimator,-0.016835,0.098418
X by facility location from Top-100 least influential (scores closest to zero). lambda=0.75,BM25Estimator,-0.008649,0.132223
X by facility location from Top-100 least influential (scores closest to zero). lambda=0.7X,BM25Estimator,0.000921,0.074816
X by facility location from Top-100 least influential (scores closest to zero). lambda=0.X,BM25Estimator,0.011829,0.111867
X by facility location from Top-100 most influential (scores with largest absolute value). lambda=0.0,BM25Estimator,-0.322618,-0.329905


### mean per k

In [35]:

linear_coder = "MSECoderProjUSimp"
compute_and_export_correlations(
    rr, rr_summary,
    subset_filter=lambda df: df[ (df["linear_coder"]==linear_coder)],
    group_cols=["estimator","model","train_dataset","train_split","test_dataset","test_split","k"],
    sort_cols=[ "k","estimator",],
    row_label_cols=[ "k","estimator"],
    filename=f"./tables/validation/per_k/{linear_coder}.tex",
    caption=f"Per k correlation of our score with validation scores for {linear_coder}"
)



### mean per estimator

In [36]:

linear_coder = "MSECoderProjUSimp"
compute_and_export_correlations(
    rr, rr_summary,
    subset_filter=lambda df: df[(df["linear_coder"]==linear_coder)],
    group_cols=["estimator","model","train_dataset","train_split","test_dataset","test_split",],
    sort_cols=[ "estimator",],
    row_label_cols=[ "estimator"],
    filename=f"./tables/validation/per_estimator/{linear_coder}.tex",
    caption=f"Per estimatior correlation of our score with validation scores for {linear_coder}"
)



,"$\rho (NMSE, \log_p)$","$\rho (NMSE, JSD)$"
estimator,,
BM25Estimator,-0.265760,-0.273444
DataInfEstimator,-0.047396,-0.048869
LESSEstimator,-0.051400,-0.059922


In [37]:

linear_coder = "MSECoderProjUSimp"
compute_and_export_correlations(
    rr, rr_summary,
    subset_filter=lambda df: df[ (df["linear_coder"]==linear_coder)],
    group_cols=["estimator","model","train_dataset","train_split","test_dataset","test_split",],
    sort_cols=[ "estimator",],
    row_label_cols=[ "estimator"],
    filename=f"./tables/validation/per_estimator/{linear_coder}.tex",
    caption=f"Per estimatior correlation of our score with validation scores for {linear_coder}"
)



,"$\rho (NMSE, \log_p)$","$\rho (NMSE, JSD)$"
estimator,,
BM25Estimator,-0.265760,-0.273444
DataInfEstimator,-0.047396,-0.048869
LESSEstimator,-0.051400,-0.059922


## per lambda

In [38]:

linear_coder = "MSECoderProjUSimp"
compute_and_export_correlations(
    rr, rr_summary,
    subset_filter=lambda df: df[(df["lambda"]!="-")& (df["linear_coder"]==linear_coder)],
    group_cols=["estimator","model","train_dataset","train_split","test_dataset","test_split","lambda"],
    sort_cols=[ "estimator","lambda"],
    row_label_cols=[ "estimator","lambda"],
    filename=f"./tables/validation/per_lambda/{linear_coder}.tex",
    caption=f"Per lambda correlation of our score with validation scores for {linear_coder}"
)

